<center><h1>Keehan_Alyssa_HW7</h1></center>

Name: Alyssa Keehan
<br>
Github Username: akeehan56
<br>
GitHub Links: [Class Repo](https://github.com/akeehan56/DSCI552) | [HW7](https://github.com/akeehan56/DSCI552/tree/main/HW7)
<br>
Email: akeehan@usc.edu
<br>
USC ID: 3429542433

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score

from sklearn.multioutput import ClassifierChain
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

### (a) Download the Anuran Calls (MFCCs) Data Set

In [4]:
df = pd.read_csv(Path("../data/Anuran Calls (MFCCs)", "Frogs_MFCCs.csv"))
df.head(10)

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,...,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,RecordID
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,...,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,Leptodactylidae,Adenomera,AdenomeraAndre,1
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,...,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,Leptodactylidae,Adenomera,AdenomeraAndre,1
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,...,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,Leptodactylidae,Adenomera,AdenomeraAndre,1
3,1.0,0.224392,0.118985,0.329432,0.372088,0.361005,0.015501,-0.194347,-0.098181,0.270375,...,-0.136009,-0.177037,-0.130498,-0.054766,-0.018691,0.023954,Leptodactylidae,Adenomera,AdenomeraAndre,1
4,1.0,0.087817,-0.068345,0.306967,0.330923,0.249144,0.006884,-0.265423,-0.172700,0.266434,...,-0.048885,-0.053074,-0.088550,-0.031346,0.108610,0.079244,Leptodactylidae,Adenomera,AdenomeraAndre,1
5,1.0,0.099704,-0.033408,0.349895,0.344535,0.247569,0.022407,-0.213767,-0.127916,0.277353,...,-0.080487,-0.130089,-0.171478,-0.071569,0.077643,0.064903,Leptodactylidae,Adenomera,AdenomeraAndre,1
6,1.0,0.021676,-0.062075,0.318229,0.380439,0.179043,-0.041667,-0.252300,-0.167117,0.220027,...,-0.046620,-0.055146,-0.085972,-0.009127,0.065630,0.044040,Leptodactylidae,Adenomera,AdenomeraAndre,1
7,1.0,0.145130,-0.033660,0.284166,0.279537,0.175211,0.005791,-0.183329,-0.158483,0.192567,...,-0.055978,-0.048219,-0.056637,-0.022419,0.070085,0.021419,Leptodactylidae,Adenomera,AdenomeraAndre,1
8,1.0,0.271326,0.027777,0.375738,0.385432,0.272457,0.098192,-0.173730,-0.157857,0.207181,...,-0.120723,-0.112607,-0.156933,-0.118527,-0.002471,0.002304,Leptodactylidae,Adenomera,AdenomeraAndre,1
9,1.0,0.120565,-0.107235,0.316555,0.364437,0.307757,0.025992,-0.294179,-0.223236,0.268435,...,-0.051073,-0.052568,-0.111338,-0.040014,0.090204,0.088025,Leptodactylidae,Adenomera,AdenomeraAndre,1


In [5]:
df.columns = df.columns.str.strip()

target_cols = ["Family", "Genus", "Species"]

record_id_cols = [
    col for col in df.columns
    if col.lower().replace("_", "").replace(" ", "") == "recordid"
]

X = df.drop(columns=target_cols + record_id_cols)
y = df[target_cols]

# split the data: 70% training, 30% test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.70,
    random_state=17
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (5036, 22) (5036, 3)
Test set: (2159, 22) (2159, 3)


### (b) Train a classifier for each label

#### (i) Research

**Answer:**

- Exact Match counts an observation as correct only when all the labels for multi-class/label classification. This measure ignores partially correct labels which could be considered harsh.
- Hamming Loss is the fraction of the labels predicted incorectly, averaged over observations. The smaller the hamming loss, the better the performance as it takes into account both the prediction and missing error. 
- Hamming Score is the reciprocal of hamming loss, the proportion of predicted correct labels to the total number of labels for that instance. It is commonly known as accuracy.

Source: [Metrics for Multilabl Classification](https://mmuratarat.github.io/2020-01-25/multilabel_classification_metrics)

#### (ii) Train a SVM for each of the labels

In [6]:
C_values = [0.1, 1, 10, 100]
sigma_values = [0.1, 0.5, 1.0, 1.5]
gamma_values = [1 / (2 * sigma**2) for sigma in sigma_values]

def fit_rbf_models(scale_features):
    models = {}
    predictions = pd.DataFrame(index=y_test.index)

    scaler = StandardScaler() if scale_features else "passthrough"

    for target in target_cols:
        pipe = ImbPipeline([
            ("scaler", scaler),
            ("model", OneVsRestClassifier(
                SVC(kernel="rbf", probability=True)
            ))
        ])

        search = GridSearchCV(
            pipe,
            param_grid={
                "model__estimator__C": C_values,
                "model__estimator__gamma": gamma_values
            },
            scoring="accuracy",
            cv=10,
            n_jobs=-1
        )

        search.fit(X_train, y_train[target])
        models[target] = search.best_estimator_
        predictions[target] = search.predict(X_test)

        print(target, search.best_params_, search.best_score_)

    return models, predictions

rbf_scaled_models, rbf_scaled_pred = fit_rbf_models(scale_features=True)
rbf_raw_models, rbf_raw_pred = fit_rbf_models(scale_features=False)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.1

KeyboardInterrupt: 

#### (iii) Repeat 1(b)ii with L1-penalized SVMs

#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

### (b) Determine which family is the majority

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

## 3. ISLR 12.6.2